# Phase 14 — Filtering and retained-test subgroups
**Owner: Shafin. CPU analysis only; no model fitting, tuning, or test-based model selection.** Use the existing **enhancer** kernel. All original inputs have been copied under `reference_inputs`; raw ENCODE/UCSC bytes are authenticated against the original Phase 9A checksums. This notebook does not download or install anything.

**Main question:** What kinds of sequences did the legacy-similarity filter remove, and how does that limit the paper's conclusions?

We distinguish the complete paired-orientation assay population, the retained filtered population, and prediction performance on retained test records. Removed sequences have no saved model predictions here: their performance cannot be inferred from composition or activity distributions.

**Subgroups fixed before this analysis:** GC bins [0,.35,.45,.55,.65,1]; CpG density below/above the training median; maximum homopolymer <6 versus ≥6; activity below training 20th percentile / middle / above training 80th percentile; chromosomes. Thresholds learned from training use only that protocol/cell's training records. Minimum reporting n=100. These are descriptive, exploratory comparisons on already examined partitions, not calibrated causal tests.

In [1]:
from pathlib import Path
import os, sys, io, json, zipfile, hashlib, itertools
import numpy as np
import pandas as pd
ROOT=Path(r"D:\Aditta Thesis\For Shafin\Shafin's Contribution").resolve()
assert ROOT.is_dir() and ROOT.name=="Shafin's Contribution"
assert Path(sys.executable).resolve()==Path(r'C:\Users\User\.conda\envs\enhancer\python.exe').resolve(), 'Select the enhancer Anaconda kernel.'
OUT=ROOT/'phase14_outputs';OUT.mkdir(exist_ok=True)
P12=ROOT/'phase12_outputs/enhancer_phase12_work'
def digest(raw):return hashlib.sha256(raw).hexdigest()
def filehash(p,algorithm='sha256'):
 with Path(p).open('rb') as f:return hashlib.file_digest(f,algorithm).hexdigest()
def member(name):
 found=[]
 for folder in ['phase12_inputs','reference_inputs']:
  for p in (ROOT/folder).glob('*.zip'):
   with zipfile.ZipFile(p) as z:
    for m in z.namelist():
     if m.split('/')[-1]==name:found.append(z.read(m))
 assert found, 'Missing copied input: '+name
 assert len({digest(b) for b in found})==1, 'Ambiguous input versions: '+name
 return found[0]
def csvmember(name):return pd.read_csv(io.BytesIO(member(name)))
def dump_json(name,obj):(OUT/name).write_text(json.dumps(obj,indent=2,sort_keys=True,allow_nan=False)+'\n',encoding='utf-8')
def write_table(name,frame):frame.to_csv(OUT/name,index=False,float_format='%.12g')
prov=json.loads((P12/'results/phase12_provenance.json').read_text())
assert prov['smoke_test'] is False
for name,h in prov['config']['input_sha256'].items():assert digest(member(name))==h,name
full=csvmember('phase9_full_activity_all_cells.csv')
splits=csvmember('phase9_frozen_splits.csv')
assert full.record_id.is_unique and splits.record_id.is_unique
DATA=full.merge(splits.drop(columns=[c for c in splits if c in full and c!='record_id']),on='record_id',validate='one_to_one')
assert len(DATA)==76833
SEEDS=[20260930,20260931,20260932]
def corr(y,p):
 y=np.asarray(y,dtype=float);p=np.asarray(p,dtype=float)
 a=y-y.mean();b=p-p.mean();den=np.sqrt(np.dot(a,a)*np.dot(b,b))
 return float(np.dot(a,b)/den) if den else float('nan')
def metrics(y,p):
 y=np.asarray(y,float);p=np.asarray(p,float);assert np.isfinite(y).all() and np.isfinite(p).all()
 e=y-p;sst=np.sum((y-y.mean())**2)
 return dict(n=len(y),pearson=corr(y,p),spearman=corr(pd.Series(y).rank().to_numpy(),pd.Series(p).rank().to_numpy()),r2=float(1-np.sum(e*e)/sst) if sst else np.nan,rmse=float(np.sqrt(np.mean(e*e))),mae=float(np.mean(abs(e))))
def internal_predictions():
 old=[csvmember(f'phase{t}_predictions_validation_test.csv').assign(protocol=t) for t in ['9b','9c']]
 return pd.concat(old+[pd.read_csv(P12/'results/phase12_internal_predictions_new_arms.csv')],ignore_index=True)
print('Environment:',sys.executable,'; full retained records:',len(DATA))

Environment: C:\Users\User\.conda\envs\enhancer\python.exe ; full retained records: 76833


## 1. Reconstruct the complete assay population and ordered filtering reasons
Raw-row exclusions (design family, chromosome, length, count reliability, orientation pairing) are counted separately from sequence filtering among complete 200 bp pairs. Each population record gets exactly one ordered reason: non-ACGT, exact/RC match to legacy, shared 15-mer with legacy, duplicate within cell, or retained. The first matching reason is used, as in Phase 9A.

In [2]:
import gzip,re,math,struct,bisect
from collections import Counter
AUDIT=json.loads(member('phase9a_audit_and_provenance.json'))
RAW=ROOT/'reference_inputs/raw_sources'
missing=[name for name in AUDIT['sources'] if not (RAW/name).is_file()]
source_status={'status':'pending' if missing else 'available','missing':missing}
if missing:print('PENDING raw population/composition reconstruction; missing copied raw files:',missing)
else:
 for name,checks in AUDIT['sources'].items():
  assert filehash(RAW/name,'md5')==checks['md5'] and filehash(RAW/name)==checks['sha256'], name
 print('All three raw sources match original Phase 9A MD5 and SHA256.')
# A minimal read-only UCSC 2bit reader avoids adding packages.
class TwoBit:
 def __init__(self,path):
  self.f=Path(path).open('rb');head=self.f.read(16)
  self.endian='<' if struct.unpack('<I',head[:4])[0]==0x1A412743 else '>'
  sig,ver,count,res=struct.unpack(self.endian+'4I',head);assert sig==0x1A412743 and ver==0
  self.offsets={};self.cache={}
  for _ in range(count):
   n=self.f.read(1)[0];name=self.f.read(n).decode();self.offsets[name]=self.uints(1)[0]
  self.lut=np.array([[ord('TCAG'[(x>>shift)&3]) for shift in [6,4,2,0]] for x in range(256)],dtype=np.uint8)
 def uints(self,n):return struct.unpack(self.endian+str(n)+'I',self.f.read(4*n)) if n else ()
 def chromosome(self,name):
  if name not in self.cache:
   self.f.seek(self.offsets[name]);size=self.uints(1)[0];n=self.uints(1)[0];starts=self.uints(n);sizes=self.uints(n)
   m=self.uints(1)[0];self.uints(m);self.uints(m);self.uints(1)
   packed=self.f.read((size+3)//4);self.cache[name]=(size,starts,sizes,packed)
  return self.cache[name]
 def sequence(self,name,start,end):
  size,ns,nl,packed=self.chromosome(name);assert 0<=start<end<=size
  b=np.frombuffer(packed,dtype=np.uint8,offset=start//4,count=(end+3)//4-start//4)
  s=bytearray(self.lut[b].ravel()[start%4:start%4+end-start].tobytes())
  j=max(0,bisect.bisect_right(ns,start)-1)
  for i in range(j,len(ns)):
   a,l=ns[i],nl[i]
   if a>=end:break
   lo=max(start,a);hi=min(end,a+l)
   if lo<hi:s[lo-start:hi-start]=b'N'*(hi-lo)
  return s.decode()
 def close(self):self.f.close()

All three raw sources match original Phase 9A MD5 and SHA256.


In [3]:
POP=None;raw_audits={}
if not missing:
 genome=TwoBit(RAW/'hg38.2bit');legacy=csvmember('stage1_candidate_manifest.csv')
 assert digest(member('stage1_candidate_manifest.csv'))==AUDIT['config']['stage1_manifest_sha256'] and len(legacy)==3268
 comp=str.maketrans('ACGT','TGCA')
 def rc(s):return s.translate(comp)[::-1]
 exact={min(s,rc(s)) for s in legacy.sequence};words=set()
 for s in legacy.sequence:
  for strand in [s,rc(s)]:words.update(strand[i:i+15] for i in range(186))
 populations=[]
 for cell,spec in AUDIT['config']['assays'].items():
  count=Counter();rows=[];pat=re.compile(re.escape(spec['prefix'])+r'\d+(?:_Reversed:)?')
  with gzip.open(RAW/spec['file'],'rt') as f:
   for line in f:
    if not line.strip() or line.startswith(('#','track','browser')):continue
    fields=line.rstrip('\n').split('\t');assert len(fields)==11;chrom,st,en,name,score,strand,fc,ic,oc,p,q=fields
    count['rows_total']+=1
    if not pat.fullmatch(name):count['other_design_name']+=1;continue
    if not re.fullmatch(r'chr(?:[1-9]|1[0-9]|2[0-2]|X|Y)',chrom):count['noncanonical_chrom']+=1;continue
    st,en=int(st),int(en)
    if en-st!=200 or st<0 or en>genome.chromosome(chrom)[0]:count['non_200bp']+=1;continue
    if strand not in ['+','-']:count['unknown_strand']+=1;continue
    assert p=='-1' and q=='-1';fc,ic,oc=map(float,[fc,ic,oc])
    if not all(map(math.isfinite,[fc,ic,oc])) or ic<=0 or oc<=0:count['unreliable_counts']+=1;continue
    rows.append((chrom,st,en,re.sub(r'_Reversed:$','',name),strand,fc,ic,oc));count['candidate_rows']+=1
  temp=pd.DataFrame(rows,columns=['chrom','start','end','design_name','strand','activity','input','output'])
  paired=temp.groupby(['chrom','start','end','design_name']).agg(n=('activity','size'),strands=('strand','nunique'),mean_log2_activity=('activity','mean'),min_input_count=('input','min'),min_output_count=('output','min')).reset_index()
  good=paired[paired.n.eq(2)&paired.strands.eq(2)].copy();count['incomplete_or_ambiguous_orientation']=len(paired)-len(good);count['complete_two_orientation_elements']=len(good)
  for key,value in count.items():assert value==AUDIT['assay_audit'][cell].get(key,0),(cell,key,value)
  good['cell_line']=cell;seqs=[];reasons=[];ids=[]
  for row in good.itertuples(index=False):
   seq=genome.sequence(row.chrom,row.start,row.end).upper();seqs.append(seq)
   ids.append(digest(f'{cell}|{row.chrom}|{row.start}|{row.end}|{row.design_name}'.encode())[:20])
   if not re.fullmatch('[ACGT]{200}',seq):reason='non_ACGT_reference'
   elif min(seq,rc(seq)) in exact:reason='exact_or_RC_legacy'
   elif any(seq[i:i+15] in words for i in range(186)):reason='shared_15mer_with_legacy'
   else:reason='retained'
   reasons.append(reason)
  good['record_id']=ids;good['sequence']=seqs;good['sequence_sha256']=[digest(s.encode()) for s in seqs];good['filter_reason']=reasons
  sorted_good=good[good.filter_reason.eq('retained')].sort_values(['sequence_sha256','chrom','start','record_id'])
  duplicates=sorted_good[sorted_good.sequence_sha256.duplicated()].index;good.loc[duplicates,'filter_reason']='within_cell_duplicate_sequences'
  for reason in ['non_ACGT_reference','exact_or_RC_legacy','shared_15mer_with_legacy','within_cell_duplicate_sequences']:
   assert int(good.filter_reason.eq(reason).sum())==AUDIT['assay_audit'][cell].get(reason,0),(cell,reason)
  retained=good[good.filter_reason.eq('retained')].set_index('record_id').sort_index();ref=DATA[DATA.cell_line.eq(cell)].set_index('record_id').sort_index()
  assert retained.index.equals(ref.index)
  assert retained.sequence.equals(ref.sequence) and retained.sequence_sha256.equals(ref.sequence_sha256)
  assert np.max(abs(retained.mean_log2_activity-ref.mean_log2_activity))<1e-10
  populations.append(good);raw_audits[cell]=dict(count)
  print('Reconstructed',cell,':',len(good),'complete pairs;',len(retained),'retained',flush=True)
 genome.close();genome.cache.clear();del genome,words,exact
 POP=pd.concat(populations,ignore_index=True)
 assert len(POP)==149632 and POP.record_id.is_unique
 POP['population']=np.where(POP.filter_reason.eq('retained'),'retained','removed')
 write_table('reconstructed_filtering_population.csv.gz',POP)
 source_status={'status':'completed','population_records':len(POP),'retained':int(POP.population.eq('retained').sum()),'removed':int(POP.population.eq('removed').sum()),'source_hashes':AUDIT['sources'],'legacy_manifest_sha256':digest(member('stage1_candidate_manifest.csv'))}
dump_json('population_reconstruction_status.json',source_status);dump_json('raw_row_exclusion_audit.json',raw_audits)

Reconstructed HepG2 : 64190 complete pairs; 33324 retained


Reconstructed K562 : 85442 complete pairs; 43509 retained


## 2. Activity distributions, DNA composition and cell-type attrition
Differences describe the screen's selection effect. They do not measure how models would perform on the removed DNA. HepG2 and K562 use different candidate-design families, so attrition differences cannot be attributed to cell biology alone. P-values are unnecessary for the central finite-population question; report counts and effect sizes.

In [4]:
def sequence_features(frame):
 frame=frame.copy();s=frame.sequence
 frame['gc_fraction']=(s.str.count('G')+s.str.count('C'))/200
 frame['cpg_density']=s.str.count('CG')/199
 probs=np.stack([s.str.count(b).to_numpy()/200 for b in 'ACGT'],axis=1)
 frame['base_entropy_bits']=-np.sum(np.where(probs>0,probs*np.log2(np.maximum(probs,1e-300)),0),axis=1)
 frame['max_homopolymer']=[max(len(m.group()) for m in re.finditer(r'A+|C+|G+|T+',seq)) for seq in s]
 return frame
summary_rows=[];activity_effects=[]
if POP is not None:
 POP=sequence_features(POP)
 attrition=POP.groupby(['cell_line','filter_reason']).size().rename('n').reset_index()
 attrition['complete_pair_population']=attrition.cell_line.map(POP.cell_line.value_counts());attrition['fraction']=attrition.n/attrition.complete_pair_population
 write_table('cell_type_filter_attrition.csv',attrition)
 for (cell,group),g in POP.groupby(['cell_line','population']):
  for feature in ['mean_log2_activity','gc_fraction','cpg_density','base_entropy_bits','max_homopolymer']:
   v=g[feature];summary_rows.append(dict(cell_line=cell,population=group,feature=feature,n=len(g),mean=v.mean(),sd=v.std(),p05=v.quantile(.05),p20=v.quantile(.2),median=v.median(),p80=v.quantile(.8),p95=v.quantile(.95)))
 for cell,g in POP.groupby('cell_line'):
  a=g[g.population.eq('retained')];b=g[g.population.eq('removed')]
  for feature in ['mean_log2_activity','gc_fraction','cpg_density','base_entropy_bits','max_homopolymer']:
   pooled=np.sqrt((a[feature].var()+b[feature].var())/2)
   activity_effects.append(dict(cell_line=cell,feature=feature,retained_minus_removed_mean=a[feature].mean()-b[feature].mean(),standardized_mean_difference=(a[feature].mean()-b[feature].mean())/pooled))
 write_table('retained_removed_distribution_summary.csv',pd.DataFrame(summary_rows));write_table('retained_removed_effect_sizes.csv',pd.DataFrame(activity_effects))
 print(attrition.round(6).to_string(index=False));print(pd.DataFrame(activity_effects).round(6).to_string(index=False))
else:print('Population distributions pending authenticated raw inputs; retained test analysis can still run.')

cell_line            filter_reason     n  complete_pair_population  fraction
    HepG2       exact_or_RC_legacy     3                     64190  0.000047
    HepG2                 retained 33324                     64190  0.519146
    HepG2 shared_15mer_with_legacy 30863                     64190  0.480807
     K562       exact_or_RC_legacy     1                     85442  0.000012
     K562                 retained 43509                     85442  0.509223
     K562 shared_15mer_with_legacy 41932                     85442  0.490766
cell_line            feature  retained_minus_removed_mean  standardized_mean_difference
    HepG2 mean_log2_activity                     0.009346                      0.013205
    HepG2        gc_fraction                     0.006648                      0.084363
    HepG2        cpg_density                     0.002625                      0.155787
    HepG2  base_entropy_bits                     0.003124                      0.098399
    HepG2    max_homo

## 3. Predeclared subgroups on retained test records
Evaluate the same retained test records for DNABERT-2, MPRA-LegNet and MPRAnn ensemble means. Error units are assay log2 activity. Correlations in narrow activity bins may drop from range restriction. These are subgroup summaries, not an uncertainty calibration model. No thresholds or model choices are selected using these test results.

The Phase 9C K562 DNABERT-2 ensemble includes the archived nearly constant seed. Its scale contraction can worsen absolute-error subgroups; notebook 13 reports a labelled healthy-seed sensitivity. Activity subgroups require measured labels and are diagnostic summaries, not deployment-time routing rules.

In [5]:
features=sequence_features(DATA);allpred=internal_predictions();subgroup_rows=[];paired_rows=[];thresholds=[]
models=['dnabert2_three_seed_mean','legnet_three_seed_mean','mprann_three_seed_mean']
for tag,cell in itertools.product(['9b','9c'],['HepG2','K562']):
 col={'9b':'split_9b','9c':'split_9a'}[tag];tr=features[features.cell_line.eq(cell)&features[col].eq('train')];te=features[features.cell_line.eq(cell)&features[col].eq('test')].copy()
 q20,q80=tr.mean_log2_activity.quantile([.2,.8]);cpg=tr.cpg_density.median()
 thresholds.append(dict(protocol=tag,cell_line=cell,training_n=len(tr),training_activity_q20=q20,training_activity_q80=q80,training_cpg_median=cpg))
 te['gc_bin']=pd.cut(te.gc_fraction,[0,.35,.45,.55,.65,1],include_lowest=True).astype(str)
 te['cpg_group']=np.where(te.cpg_density<=cpg,'at_or_below_training_median','above_training_median')
 te['homopolymer_group']=np.where(te.max_homopolymer>=6,'run_at_least_6','run_below_6')
 te['activity_group']=pd.cut(te.mean_log2_activity,[-np.inf,q20,q80,np.inf],labels=['below_training_q20','middle','above_training_q80']).astype(str)
 pred=allpred[allpred.protocol.eq(tag)&allpred.cell_line.eq(cell)&allpred.split.eq('test')&allpred.model.isin(models)]
 wide=pred.pivot(index='record_id',columns='model',values='prediction');assert set(wide.index)==set(te.record_id)
 te=te.set_index('record_id').join(wide,validate='one_to_one')
 for grouping in ['gc_bin','cpg_group','homopolymer_group','activity_group','chrom']:
  for label,g in te.groupby(grouping):
   if len(g)<100:
    paired_rows.append(dict(protocol=tag,cell_line=cell,subgroup_type=grouping,subgroup=str(label),n=len(g),status='suppressed_n_below_100'));continue
   for model in models:subgroup_rows.append(dict(protocol=tag,cell_line=cell,subgroup_type=grouping,subgroup=str(label),model=model,**metrics(g.mean_log2_activity,g[model]),mean_target=float(g.mean_log2_activity.mean()),mean_prediction=float(g[model].mean()),mean_signed_error=float((g[model]-g.mean_log2_activity).mean())))
   ae=abs(g[models].sub(g.mean_log2_activity,axis=0));paired_rows.append(dict(protocol=tag,cell_line=cell,subgroup_type=grouping,subgroup=str(label),n=len(g),status='descriptive',dnabert2_minus_legnet_mae=float((ae[models[0]]-ae[models[1]]).mean()),dnabert2_minus_mprann_mae=float((ae[models[0]]-ae[models[2]]).mean())))
subgroup=pd.DataFrame(subgroup_rows);paired=pd.DataFrame(paired_rows)
write_table('retained_test_subgroup_metrics.csv',subgroup);write_table('paired_subgroup_error_differences.csv',paired);write_table('training_only_subgroup_thresholds.csv',pd.DataFrame(thresholds))
print('Positive paired MAE differences mean the CNN has smaller error.')
print(paired[paired.subgroup_type.ne('chrom')].round(5).to_string(index=False))

Positive paired MAE differences mean the CNN has smaller error.
protocol cell_line     subgroup_type                    subgroup     n                 status  dnabert2_minus_legnet_mae  dnabert2_minus_mprann_mae
      9b     HepG2            gc_bin              (-0.001, 0.35]   262            descriptive                    0.03915                    0.02149
      9b     HepG2            gc_bin                (0.35, 0.45]  3281            descriptive                    0.04152                    0.04065
      9b     HepG2            gc_bin                (0.45, 0.55]  3802            descriptive                    0.03097                    0.02849
      9b     HepG2            gc_bin                (0.55, 0.65]  1438            descriptive                    0.01682                    0.00971
      9b     HepG2            gc_bin                 (0.65, 1.0]   265            descriptive                   -0.00025                    0.00085
      9b     HepG2         cpg_group       above

## 4. Save limitations and evidence
The 15-mer screen removes overlap with the legacy benchmark and selects a narrower population. It is not proof that every form of sequence homology is absent. Population attrition and subgroup prediction accuracy address different questions. Performance on removed candidates, other cell types, native chromatin and longer sequences remains open.

In [6]:
status={'status':'completed' if POP is not None else 'partial_raw_population_pending','phase12_protocol_sha256':prov['protocol_sha256'],'population_reconstruction':source_status,'subgroup_metric_rows':len(subgroup),'minimum_subgroup_n':100,'subgroups_fixed_before_analysis':['fixed_GC_bins','training_median_CpG','homopolymer_at_least_6','training_activity_20_80_percentiles','chromosome'],'removed_sequence_performance':'unavailable: no saved predictions on removed candidates; no training or inference launched','causal_limits':'Cells have different candidate designs; descriptive contrasts do not isolate biology or prove sequence sufficiency/insufficiency','multiple_comparisons':'Descriptive exploration, no significance claims or tuning'}
dump_json('filtering_analysis_summary.json',status)
if POP is not None:
 effect=pd.DataFrame(activity_effects)
 lines=['# Filtering evidence — Shafin, 2026-10-12','','Complete paired-orientation population: '+str(len(POP))+'. Retained: 76,833. Removed: '+str(int(POP.population.eq('removed').sum()))+'.','', 'Counts and features reproduce the original frozen retained sequences exactly.','', 'Mean shifts (retained minus removed):', effect.round(6).to_string(index=False),'','Scope: retained 200 bp candidates in these two assay/design families; removed-sequence performance is unknown. No model tuning or additional training.']
 (OUT/'filtering_findings.md').write_text('\n'.join(lines)+'\n',encoding='utf-8')
print(json.dumps(status,indent=2))

{
  "status": "completed",
  "phase12_protocol_sha256": "1506ebc3e38db096dfa16fcfab1b0e29e8a7cefaa8f8255a678bdce012bbffcf",
  "population_reconstruction": {
    "status": "completed",
    "population_records": 149632,
    "retained": 76833,
    "removed": 72799,
    "source_hashes": {
      "ENCFF631LHA.bed.gz": {
        "md5": "847512c673c75874e84200f84899f880",
        "sha256": "c2e2a6014375197c7aac96eb68b9dc4a917d671ba415ca8b38edb287b7695d4b"
      },
      "ENCFF774DYO.bed.gz": {
        "md5": "87b3fb363f64aeb2b5f893ff98b47db3",
        "sha256": "a95ecedfab3b3074075f51efeca583d5ba138ce3288a2eba3e265b627eb4427f"
      },
      "hg38.2bit": {
        "md5": "dcc3ea27079aa6dc3f9deccd7275e0f8",
        "sha256": "1f67aaa17a77b327738fe750ab37430a85dddf73c7d9189385ad1839259acec0"
      }
    },
    "legacy_manifest_sha256": "e295155300fafc34528bc1868c6fad6ec78f3912c5b07e034710e81c3b4ac4b3"
  },
  "subgroup_metric_rows": 306,
  "minimum_subgroup_n": 100,
  "subgroups_fixed_before_anal

## 5. Export reusable population figures
Figures are descriptive and keep both cell/design families separate. The subgroup analysis above uses retained test records only.

In [7]:

os.environ['MPLCONFIGDIR']=str(OUT/'matplotlib_cache')
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
if POP is not None:
 fig,axes=plt.subplots(2,2,figsize=(10,7),layout='constrained')
 colors={'retained':'#2364AA','removed':'#D97732'}
 for row,cell in enumerate(['HepG2','K562']):
  sub=POP[POP.cell_line.eq(cell)]
  for col,feature in enumerate(['mean_log2_activity','gc_fraction']):
   bins=np.linspace(sub[feature].min(),sub[feature].max(),55)
   for population,g in sub.groupby('population'):
    axes[row,col].hist(g[feature],bins=bins,weights=np.ones(len(g))/len(g),histtype='step',linewidth=1.8,color=colors[population],label=f'{population} (n={len(g):,})')
   axes[row,col].set_title(cell+' — '+('assayed activity' if col==0 else 'GC composition'))
   axes[row,col].set_xlabel('Mean log2 output/input' if col==0 else 'GC fraction');axes[row,col].set_ylabel('Fraction of population per bin');axes[row,col].legend(frameon=False,fontsize=8)
 fig.suptitle('Selection effects of the legacy-similarity screen',fontsize=14)
 fig.savefig(OUT/'retained_removed_activity_and_gc.png',dpi=220);fig.savefig(OUT/'retained_removed_activity_and_gc.pdf');plt.close(fig)
 print('Saved population comparison figure as PNG and PDF.')
else:print('Population figures pending complete authenticated reference copy.')


Saved population comparison figure as PNG and PDF.
